# Perche i residui non separanoLa replica del Capitolo 6 si ferma a rapporti di 1,04 e 1,15 fra il residuo dei guasti equello dei sani, contro i 3,71 e 4,61 dichiarati dal paper. Il punto in cui il metodo cedesta prima della classificazione: se il residuo non separa, la rete a valle non ha materiasu cui lavorare.Questo notebook tiene i dati fermi e muove una cosa alla volta nel modo in cuil'autoencoder e costruito e addestrato. Se la responsabilita e della tecnica, deve saltarefuori cosi.Restano invariati rispetto alla replica: i 17 cuscinetti della Tabella 2, il regime`N15_M07_F10`, il canale `phase_current_1`, i segmenti da un secondo divisi in 25 frame da2560 campioni, e i 2560 frame sani per l'addestramento divisi 2048/512.Le prove sono sei. La prima non addestra nulla: guarda il residuo che gia abbiamo, ma persingolo esemplare invece che per classe.

In [ ]:
!apt-get -qq update && apt-get -qq install -y unrar
!pip -q install requests scipy

In [ ]:
import os, sys, subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

try:
    from google.colab import drive
    drive.mount('/content/drive')
except ImportError:
    pass

REPO = 'https://github.com/matpaol/MacchineEdAzionamentiExam'
possibili = ['.', '..', '../codice',
             '/content/drive/MyDrive/MacchineEdAzionamentiExam',
             '/content/MacchineEdAzionamentiExam']
percorso_codice = None
for c in possibili:
    if os.path.exists(os.path.join(c, 'funzioni.py')):
        percorso_codice = os.path.abspath(c)
        break
if percorso_codice is None:
    print('codice non trovato in locale, clono il repo')
    subprocess.run(['git', 'clone', '-q', REPO, '/content/MacchineEdAzionamentiExam'], check=True)
    percorso_codice = '/content/MacchineEdAzionamentiExam'
sys.path.insert(0, percorso_codice)
import config
import funzioni as f

f.stile_grafici()
P = config.percorsi(sottocartella='03_indagine_replica')
dev = f.dispositivo()
seme = 0

print('codice da', percorso_codice)
print('risultati in', P['risultati'])
print('dispositivo', dev)

## I dati, gli stessi della replicaSi ricostruiscono i 1355 segmenti e i 33 875 frame esattamente come nel notebook dellareplica, e si sorteggiano gli stessi 2560 frame sani con lo stesso seme. Tutte le prove cheseguono partono da qui: cambia una cosa per volta, e nient'altro.

In [ ]:
cuscinetti = config.CUSCINETTI_PAPER
regime = config.REGIME_PRINCIPALE

f.estrai_misure(cuscinetti, P['raw'], P['estratti'])
inv = f.inventario(cuscinetti, P['estratti'], regimi=[regime]).reset_index(drop=True)
segmenti, anagrafica = f.costruisci_segmenti(inv, segmenti_per_registrazione=4)

lunghezza_frame = f.lunghezza_frame(config.REGIMI[regime]['rpm'])
frame_per_segmento = segmenti.shape[1] // lunghezza_frame
frame = segmenti.reshape(-1, lunghezza_frame)

classe_del_frame = np.repeat(anagrafica['classe'].values, frame_per_segmento)
cuscinetto_del_frame = np.repeat(anagrafica['cuscinetto'].values, frame_per_segmento)
classe_del_segmento = anagrafica['classe'].values

rng = np.random.default_rng(seme)
scelti = rng.choice(np.flatnonzero(classe_del_frame == 0), size=2560, replace=False)
rng.shuffle(scelti)
frame_train, frame_val = frame[scelti[:2048]], frame[scelti[2048:]]

print(len(segmenti), 'segmenti |', len(frame), 'frame da', lunghezza_frame, 'campioni')
print('frame sani per il DAE:', len(scelti), '-> addestramento', len(frame_train),
      '| validazione', len(frame_val))
print('campioni sotto il pavimento della SELU:',
      round(100 * float(np.mean(frame < config.PAVIMENTO_SELU)), 2), '%')

## Come si misura una provaOgni prova addestra un autoencoder, calcola il residuo su tutti i frame e ne ricava semprele stesse grandezze. Due sono quelle che contano davvero.Il **rapporto** fra il residuo medio dei guasti e quello dei sani e la grandezza che ilpaper riporta, e serve per il confronto diretto. Ha pero un difetto: mette a confronto duemedie e non dice nulla su quanto le due distribuzioni si sovrappongano.L'**AUC** dice proprio questo. E la probabilita che, preso a caso un segmento guasto e unosano, il guasto abbia residuo piu alto. Vale 0,5 quando le due distribuzioni coincidono e 1quando sono separate del tutto, e non dipende dalla scala del segnale: due prove conampiezze diverse restano confrontabili.Quando il segnale viene scalato prima di entrare nella rete, il residuo viene riportato inampere dividendo per lo stesso fattore, cosi tutte le righe della tabella finale parlanodella stessa unita di misura.

In [ ]:
def prova(etichetta, fattore_scala=1.0, uscita_selu=True, dimensioni=None,
          epoche=500, lotto=256, passo=3e-4, weight_decay=0.0, pazienza=None,
          tieni_residui=False):
    """Addestra un autoencoder e ne misura la separazione. Una variabile alla volta."""
    print(etichetta)
    modello, curva_train, curva_val = f.addestra_dae(
        frame_train * fattore_scala, frame_val * fattore_scala,
        uscita_selu=uscita_selu, dimensioni=dimensioni, epoche=epoche, lotto=lotto,
        passo=passo, weight_decay=weight_decay, pazienza=pazienza,
        seme=seme, dev=dev, stampa_ogni=250)

    residui = f.calcola_residui(modello, frame * fattore_scala, dev=dev) / fattore_scala
    mse = f.mse_per_frame(residui)
    per_classe = [float(np.mean(mse[classe_del_frame == c])) for c in (0, 1, 2)]
    mse_segmento = mse.reshape(len(segmenti), frame_per_segmento).mean(axis=1)
    auc = f.auc_residuo(mse_segmento, classe_del_segmento)

    riga = {'prova': etichetta,
            'residuo_sano': per_classe[0],
            'residuo_esterno': per_classe[1],
            'residuo_interno': per_classe[2],
            'rapporto_esterno': per_classe[1] / per_classe[0],
            'rapporto_interno': per_classe[2] / per_classe[0],
            'auc_esterno': auc['esterno'],
            'auc_interno': auc['interno'],
            'auc_media': auc['media'],
            'parametri': f.conta_parametri(modello),
            'epoche': len(curva_val),
            'epoca_minimo': int(np.argmin(curva_val)) + 1}

    print('   rapporti {:.3f} / {:.3f}   AUC {:.4f} / {:.4f}   media {:.4f}'.format(
        riga['rapporto_esterno'], riga['rapporto_interno'],
        riga['auc_esterno'], riga['auc_interno'], riga['auc_media']))
    print()

    esito = {'riga': riga, 'curve': (curva_train, curva_val)}
    if tieni_residui:
        esito['mse_frame'] = mse
        esito['mse_segmento'] = mse_segmento
    return esito


risultati = []
curve = {}

## Il riferimentoE la replica letterale: SELU anche sull'uscita, segnale non trattato, collo di bottiglia a32, 500 epoche fisse. Ogni prova successiva si confronta con questa riga.

In [ ]:
riferimento = prova('riferimento (replica letterale)', tieni_residui=True)
risultati.append(riferimento['riga'])
curve['riferimento'] = riferimento['curve']

## Prova 1. Il residuo, cuscinetto per cuscinettoQuesta non addestra nulla: usa il residuo appena calcolato, ma lo guarda per singoloesemplare invece che per classe.La domanda viene prima di tutte le altre. Il paper riporta tre medie, una per classe. Sepero esemplari della stessa classe producessero residui molto diversi fra loro, e se leclassi si sovrapponessero, allora il residuo non starebbe ordinando i cuscinetti per statoma per identita. In quel caso raffinare la tecnica sarebbe inutile, perche lo strumentosarebbe puntato sulla cosa sbagliata.Non e una curiosita accademica: e la misura di quanto il metodo sia utilizzabile infabbrica, dove il cuscinetto su cui si installa il sistema non e nessuno di quelli su cui estato addestrato.

In [ ]:
tabella_cuscinetti = f.residuo_per_cuscinetto(riferimento['mse_frame'],
                                              cuscinetto_del_frame,
                                              classe_del_frame)
print(tabella_cuscinetti.round(5).to_string(index=False))
print()

sani = tabella_cuscinetti[tabella_cuscinetti['classe'] == 'normale']['media']
guasti = tabella_cuscinetti[tabella_cuscinetti['classe'] != 'normale']['media']
print('effetto della classe:  media guasti / media sani = {:.3f}'.format(
    guasti.mean() / sani.mean()))
print('dispersione fra esemplari: massimo / minimo = {:.3f}'.format(
    tabella_cuscinetti['media'].max() / tabella_cuscinetti['media'].min()))
print()
print('cuscinetto con il residuo piu basso in assoluto:',
      tabella_cuscinetti.iloc[0]['cuscinetto'],
      '(' + tabella_cuscinetti.iloc[0]['classe'] + ')')
print('guasti con residuo piu basso del peggiore dei sani:',
      int((guasti < sani.max()).sum()), 'su', len(guasti))

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))
ordine = tabella_cuscinetti['cuscinetto'].tolist()
dati = [riferimento['mse_frame'][cuscinetto_del_frame == c] for c in ordine]
colori = [f.COLORI[cl] for cl in tabella_cuscinetti['classe']]

disegno = ax.boxplot(dati, patch_artist=True, widths=0.6, showfliers=False)
for corpo, colore in zip(disegno['boxes'], colori):
    corpo.set_facecolor(colore); corpo.set_alpha(0.8); corpo.set_edgecolor(f.COLORI['scuro'])
for chiave in ['medians', 'whiskers', 'caps']:
    for linea in disegno[chiave]:
        linea.set_color(f.COLORI['scuro'])

ax.set_xticks(range(1, len(ordine) + 1))
ax.set_xticklabels(ordine, rotation=45, ha='right')
ax.set_ylabel('residuo del frame')
ax.set_title('Residuo per singolo cuscinetto, ordinati per valore medio', fontsize=10.5)
for cl in config.NOMI_CLASSI:
    ax.plot([], [], 's', color=f.COLORI[cl], label=cl)
ax.legend(fontsize=8)

f.salva_figura(fig, 'residuo_per_cuscinetto', P['figure'])
plt.show()

## Prova 2. Scalare il segnale in ingressoIl paper prescrive la SELU anche sull'ultimo strato, e quella funzione non puo produrrevalori inferiori a $-1{,}7581$, mentre la corrente arriva a $-3$~A. Una parte del segnale eirraggiungibile per costruzione, e il residuo che ne deriva misura il taglio invece dellacondizione del cuscinetto.Si puo togliere il troncamento in due modi. Il primo e cambiare l'attivazione dell'ultimostrato, che pero significa cambiare l'architettura dell'articolo. Il secondo e lasciare larete com'e e spostare il segnale dentro il campo della funzione, moltiplicandolo per unacostante ricavata dai soli frame sani di addestramento.Il secondo modo e preferibile perche non tradisce il paper. Il fattore porta il picco deisani a 1,5, cioe sotto la soglia con un margine, perche i cuscinetti guasti hanno picchileggermente piu alti e il fattore non li ha visti.

In [ ]:
fattore, massimo_sani = f.scala_globale(frame_train)
minimo_scalato = float(np.min(frame)) * fattore

print('massimo assoluto dei sani di addestramento:', round(massimo_sani, 4))
print('fattore di scala:', round(fattore, 6))
print('minimo su TUTTI i segnali scalati:', round(minimo_scalato, 6),
      '| pavimento:', round(config.PAVIMENTO_SELU, 4))
print('campioni ancora sotto il pavimento:',
      round(100 * float(np.mean(frame * fattore < config.PAVIMENTO_SELU)), 3), '%')
print()

for etichetta, argomenti in [
        ('uscita lineare', dict(uscita_selu=False)),
        ('scala globale in ingresso', dict(fattore_scala=fattore))]:
    esito = prova(etichetta, **argomenti)
    risultati.append(esito['riga'])
    curve[etichetta] = esito['curve']

## Prova 3. Stringere il collo di bottigliaL'ipotesi qui e opposta a quella della prova precedente: il DAE avrebbe capacitasufficiente per ricostruire bene anche i segnali guasti, e questo appiattirebbe il residuo.Comprimendo di piu, la rete dovrebbe essere costretta a imparare soltanto la strutturaessenziale dello stato sano, e a ricostruire peggio tutto il resto.Si prova con 8 e con 2, lasciando invariata tutta la catena che precede. Il valore 4 vienesaltato perche sarebbe una configurazione intermedia fra le due e non aggiungerebbeinformazione.Il rischio da tenere d'occhio e che un collo troppo stretto peggiori la ricostruzione anchedei sani: in quel caso il residuo cresce dappertutto, il fondo sale insieme al segnale, e ilrapporto non si muove.

In [ ]:
for collo in [8, 2]:
    dimensioni = [lunghezza_frame, 1280, 640, 320, 128, collo, 128, 320, 640, 1280,
                  lunghezza_frame]
    esito = prova('collo di bottiglia ' + str(collo), dimensioni=dimensioni)
    risultati.append(esito['riga'])
    curve['collo ' + str(collo)] = esito['curve']

## Prova 4. Fermarsi quando la validazione smette di migliorareIl paper addestra per 500 epoche fisse, senza guardare la validazione. Se la rete continuaa imparare oltre il necessario, finisce per generalizzare anche a caratteristiche presentinei segnali guasti, e il residuo si appiattisce.Si confrontano due pazienze. La validazione contiene solo frame sani diversi da quelli diaddestramento, e alla fine si ripristinano i pesi dell'epoca migliore, non quellidell'ultima eseguita.Questa prova ha anche una ricaduta pratica: se l'arresto anticipato da risultati equivalentialle 500 epoche fisse, diventa l'impostazione predefinita della fase successiva, dove ildataset e dieci volte piu grande e addestrare a lungo costerebbe ore.

In [ ]:
for pazienza in [20, 100]:
    esito = prova('arresto anticipato, pazienza ' + str(pazienza), pazienza=pazienza)
    risultati.append(esito['riga'])
    curve['pazienza ' + str(pazienza)] = esito['curve']

## Prova 5. Penalizzare i pesi grandiTerza versione della stessa idea, dal lato della regolarizzazione. Aggiungendo alla funzionedi costo una penalita proporzionale alla grandezza dei pesi si scoraggia la retedall'adattarsi ai dettagli dei singoli frame visti, e la si spinge verso unarappresentazione piu generale dello stato sano.Si prova un solo valore, deciso prima di eseguire. Non e una ricerca del parametro migliore:serve solo a capire se la regolarizzazione abbia un effetto riconoscibile sulla separazione.

In [ ]:
esito = prova('weight decay 1e-4', weight_decay=1e-4)
risultati.append(esito['riga'])
curve['weight decay'] = esito['curve']

## Prova 6. Lotto e passo di apprendimentoQuesta prova non risponde a una domanda sulla macchina: e un controllo su di noi.Il lotto e quanti frame la rete guarda prima di aggiornare i pesi una volta; il passo equanto grande e ciascun aggiornamento. Nessuno dei due cambia cosa la rete puo imparare:cambiano soltanto la strada per arrivarci.Se il rapporto fra guasti e sani si muovesse cambiando la dimensione del lotto, vorrebbedire che quel rapporto non sta misurando una proprieta del cuscinetto. E la ragione per cuitutte le prove vengono lette anche con l'AUC.

In [ ]:
for etichetta, argomenti in [('lotto 512', dict(lotto=512)),
                             ('passo 1e-3', dict(passo=1e-3))]:
    esito = prova(etichetta, **argomenti)
    risultati.append(esito['riga'])
    curve[etichetta] = esito['curve']

## Il quadro d'insieme

In [ ]:
confronto = pd.DataFrame(risultati)
confronto['rapporto_paper_esterno'] = config.PAPER_RESIDUO['esterno'] / config.PAPER_RESIDUO['normale']
confronto['rapporto_paper_interno'] = config.PAPER_RESIDUO['interno'] / config.PAPER_RESIDUO['normale']

colonne = ['prova', 'residuo_sano', 'rapporto_esterno', 'rapporto_interno',
           'auc_esterno', 'auc_interno', 'auc_media', 'epoche', 'epoca_minimo']
print(confronto[colonne].round(4).to_string(index=False))
print()
print('per confronto, i rapporti dichiarati dal paper: {:.3f} e {:.3f}'.format(
    confronto['rapporto_paper_esterno'].iloc[0], confronto['rapporto_paper_interno'].iloc[0]))
print()

migliore = confronto.loc[confronto['auc_media'].idxmax()]
print('separazione migliore:', migliore['prova'],
      '-> AUC media {:.4f}, rapporti {:.3f} / {:.3f}'.format(
          migliore['auc_media'], migliore['rapporto_esterno'], migliore['rapporto_interno']))
print('escursione del rapporto esterno fra tutte le prove: da {:.3f} a {:.3f}'.format(
    confronto['rapporto_esterno'].min(), confronto['rapporto_esterno'].max()))

In [ ]:
fig, assi = plt.subplots(1, 2, figsize=(12, 4.4))
ordine = confronto.sort_values('auc_media')
posizioni = np.arange(len(ordine))

assi[0].barh(posizioni, ordine['auc_media'], color=f.COLORI['nostro'])
assi[0].axvline(0.5, color=f.COLORI['neutro'], ls='--', lw=1)
assi[0].set_yticks(posizioni); assi[0].set_yticklabels(ordine['prova'], fontsize=8)
assi[0].set_xlim(0.45, 1.0)
assi[0].set_xlabel('AUC media'); assi[0].set_title('Quanto separa ciascuna prova', fontsize=10)

assi[1].barh(posizioni - 0.2, ordine['rapporto_esterno'], 0.4,
             label='esterno / sano', color=f.COLORI['esterno'])
assi[1].barh(posizioni + 0.2, ordine['rapporto_interno'], 0.4,
             label='interno / sano', color=f.COLORI['interno'])
assi[1].axvline(confronto['rapporto_paper_esterno'].iloc[0], color=f.COLORI['accento'],
                ls='--', lw=1, label='paper, esterno')
assi[1].set_yticks(posizioni); assi[1].set_yticklabels([])
assi[1].set_xlabel('rapporto sul sano')
assi[1].set_title('Rapporti, contro il valore dichiarato', fontsize=10)
assi[1].legend(fontsize=7)

f.salva_figura(fig, 'confronto_prove', P['figure'])
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
for nome in ['riferimento', 'scala globale in ingresso', 'collo 2', 'pazienza 20']:
    if nome in curve:
        ax.plot(curve[nome][1], lw=1.1, label=nome)
ax.set_yscale('log')
ax.set_xlabel('epoca'); ax.set_ylabel('errore di validazione')
ax.set_title('Andamento della validazione per alcune prove', fontsize=10.5)
ax.legend(fontsize=8)

f.salva_figura(fig, 'curve_validazione', P['figure'])
plt.show()

In [ ]:
f.salva_tabella(confronto, 'confronto_prove', P['tabelle'])
f.salva_tabella(tabella_cuscinetti, 'residuo_per_cuscinetto', P['tabelle'])

for cartella in [P['figure'], P['tabelle']]:
    for nome in sorted(os.listdir(cartella)):
        percorso = os.path.join(cartella, nome)
        if os.path.isfile(percorso):
            print(nome, round(os.path.getsize(percorso) / 1e6, 2), 'MB')